<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c04-header.png" alt="Nextia Learning · SQL and Data Preparation for AI" width="100%">

# Set up a practice database

**[Set up a practice database](https://learning.nextia-ai.com/courses/sql/m01/set-up-a-practice-database/)** · SQL and Data Preparation for AI · Module 1, lesson 3 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** build the practice database with two small Python scripts, run saved SQL files, reset the database, and read the schema of each table in SQL.

| | |
|---|---|
| **Time** | About 50 minutes |
| **Needs** | An internet connection for the first cell. No account and no install: only Python's standard library, and pandas to show tables if it is there (it is in Colab and Kaggle). |
| **You should know** | Keys, grain and the checks of the export, from [Relational concepts](https://learning.nextia-ai.com/courses/sql/m01/relational-concepts/). |
| **You do not need** | Anything on your computer. This notebook shows each step of the setup here. To make the project on your computer, follow the [lesson page](https://learning.nextia-ai.com/courses/sql/m01/set-up-a-practice-database/#build-the-database) in a terminal. |
| **Tested** | 2026-10-07, Python 3.14.6 (SQLite 3.50.4), in Jupyter on macOS, from a fresh start (*Restart and run all*) |

### How to use this notebook

The lesson has two parts. On your computer, you make the project folder `~/projects/ticket-data` in a terminal: the commands are on the [lesson page](https://learning.nextia-ai.com/courses/sql/m01/set-up-a-practice-database/). This notebook does the same steps here, in a folder `ticket-data`, and shows what each script does. It also has the module practice. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C04/M01-L03-set-up-a-practice-database/set-up-a-practice-database-solution.ipynb).

## Setup: get the practice data

Run the next cell. It downloads the three files of the Larkfield help-desk export (about 200 KB), checks each file's checksum, and builds the practice database, `larkfield.db`. It works in a folder `ticket-data`, laid out like your project from Module 1, so the code of the lesson runs here unchanged. It needs an internet connection and no account, and takes about 10 seconds.

You should see `Ready: customers 240, tickets 1417, outcomes 1874`. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings (it needs a verified phone number).
- `wrong checksum`: delete the folder `ticket-data` (in Colab: the **Files** panel on the left), then run the cell again.

This is the same database that `get_data.py` builds in [Set up a practice database](https://learning.nextia-ai.com/courses/sql/m01/set-up-a-practice-database/#build-the-database). The data is made up for this course; it has no real people.

In [ ]:
# Setup: download the Larkfield export, check it, and build larkfield.db in
# ticket-data/. Standard library only: the same steps as get_data.py.
import csv, hashlib, json, os, sqlite3, subprocess, sys, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/C04/"
FILES = {
    "customers.csv": "7b437642f0095524f8311f67a7caeb17100d23eed92618b9cf012cd3d3b79993",
    "tickets.csv": "9c47e08cf2ec5977733d32cc21eb97d0dd01b021d1f5fbf7ec31fef6279b1a28",
    "outcomes.csv": "67b4d9d9275b32061a8888050c351df7314054c11ee1ce29e856c1d5403d0653",
}
# The notebook works in its own folder ticket-data/, marked with a file, so it
# never writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
PROJECT = Path("ticket-data").resolve()
PROJECT.mkdir(exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def download(url, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

def fetch(name, expected):
    path = PROJECT / "data" / "raw" / name
    if not (path.exists() and hashlib.sha256(path.read_bytes()).hexdigest() == expected):
        download(LABS + "data/" + name, path)
        if hashlib.sha256(path.read_bytes()).hexdigest() != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder ticket-data and run again.")
    return path

DB = PROJECT / "larkfield.db"
if not DB.exists():
    paths = {name: fetch(name, sha) for name, sha in FILES.items()}
    build = sqlite3.connect(DB)
    build.executescript("""
        CREATE TABLE customers (customer_id TEXT, segment TEXT, region TEXT, joined_on TEXT);
        CREATE TABLE tickets (ticket_id TEXT, customer_id TEXT, created_at TEXT, channel TEXT, team TEXT,
            priority INTEGER, order_value REAL, word_count INTEGER, first_reply_minutes INTEGER,
            priority_now INTEGER, closed_at TEXT);
        CREATE TABLE outcomes (outcome_id TEXT, ticket_id TEXT, recorded_at TEXT, status TEXT,
            resolution_code TEXT, csat INTEGER);
        CREATE TABLE snapshot (name TEXT, value TEXT);
        INSERT INTO snapshot VALUES ('source', 'Larkfield help-desk export (synthetic, Nextia Learning C04)'),
            ('exported_at', '2026-07-01 06:00:00'), ('data_version', '1.0');
    """)
    for name, path in paths.items():
        with open(path, newline="", encoding="utf-8") as f:
            rows = list(csv.reader(f))
        marks = ", ".join("?" for _ in rows[0])
        build.executemany(f"INSERT INTO {name[:-4]} ({', '.join(rows[0])}) VALUES ({marks})",
                          [[None if v == "" else v for v in row] for row in rows[1:]])
    build.commit()
    build.close()

os.chdir(PROJECT)  # work in the project folder, as in the lesson
for folder in ("queries", "scripts", "reports"):
    Path(folder).mkdir(exist_ok=True)
# Read-only, as q.py: a query cannot change the data by mistake.
con = sqlite3.connect("file:larkfield.db?mode=ro", uri=True)
counts = [con.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0] for t in ("customers", "tickets", "outcomes")]
print("Ready: customers {}, tickets {}, outcomes {}".format(*counts))

## Setup: the helpers

The next cell defines small functions that the lesson uses. You do not need to read them, but you can.

- `sql(query)` runs a query and shows the result as a table, with the number of rows, like `q.py` and the boxes on the lesson page. It shows the first 20 rows; `sql(query, limit=None)` shows all of them.
- `check(query, ...)` compares the result of your query with the expected result, and says what is different. It does not show the answer.
- `expect(what, yours, expected)` compares one of your numbers with the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: show a query result, and check an answer without showing it.
try:
    import pandas as pd
except ImportError:
    pd = None

def run(query, db=None):
    cursor = (db or con).execute(query)
    columns = [d[0] for d in cursor.description] if cursor.description else []
    return columns, cursor.fetchall()

def sql(query, db=None, limit=20):
    """Run a query and show its result as a table, with the row count."""
    columns, rows = run(query, db)
    if not columns:
        print("No result: the cell has no SELECT yet. Write your query between the triple quotes, then run the cell again.")
        return
    shown = rows if limit is None else rows[:limit]
    if pd is not None:
        # Show NULL as NULL, as q.py and the boxes on the lesson page do.
        table = pd.DataFrame(shown, columns=columns, dtype=object)
        display(table.where(table.notna(), "NULL"))
    else:
        print("  ".join(columns))
        for row in shown:
            print("  ".join("NULL" if v is None else str(v) for v in row))
    more = f" (showing {len(shown)}; use limit=None to see all)" if len(shown) < len(rows) else ""
    print(f"{len(rows)} row{'s' if len(rows) != 1 else ''}{more}")

def fingerprint(rows, ordered=False):
    cells = [tuple("NULL" if v is None else f"{v:.6f}" if isinstance(v, float) else str(v) for v in r) for r in rows]
    return hashlib.sha256(repr(cells if ordered else sorted(cells)).encode()).hexdigest()[:16]

def check(query, rows, columns, expected, ordered=False, db=None):
    """Compare your result with the expected one: the column count, the row
    count, then the values (as a fingerprint, so the answer stays hidden)."""
    try:
        got_columns, got = run(query, db)
    except sqlite3.Error as error:
        print(f"Not yet: the query has an error: {error}")
        return
    if len(got_columns) != columns:
        print(f"Not yet: your result has {len(got_columns)} columns; the task asks for {columns}.")
    elif len(got) != rows:
        print(f"Not yet: your result has {len(got)} rows; the task expects {rows}.")
    elif fingerprint(got, ordered) != expected:
        print("Not yet: the number of rows is right, but some values" + (" or their order" if ordered else "") + " are different.")
    else:
        print("Check passed.")

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    same = abs(yours - expected) <= tolerance * max(1, abs(expected)) if isinstance(expected, float) else yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours}, which is not the expected value.")

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. Two routes to one database

On your computer, two small scripts build the practice database and query it:

| Script | What it does |
|---|---|
| `get_data.py` | Downloads the three CSV files of the export into `data/raw/`, checks each file's checksum, and loads them into `larkfield.db` |
| `q.py` | Runs one SQL query, from a `.sql` file or typed in quotes, and prints the result as a table |

The setup cell of this notebook did the same steps as `get_data.py`, in a folder `ticket-data`. The next sections look at each step. Then you run the two real scripts here.

> **On your computer.** You type the commands in a terminal, in `~/projects/ticket-data`: see [Build the database](https://learning.nextia-ai.com/courses/sql/m01/set-up-a-practice-database/#build-the-database). On macOS and Linux, type `python3`; on Windows, see [Which Python command to type](https://learning.nextia-ai.com/courses/sql/m01/set-up-a-practice-database/#which-python-command-to-type).

## 2. Step 1: download the files

Run the cell. It lists the files in `data/raw/` with their sizes.

In [ ]:
for path in sorted(Path("data/raw").iterdir()):
    print(f"{path.name:<15} {path.stat().st_size:>7} bytes")

> **Check.** You should see `customers.csv`, `outcomes.csv` and `tickets.csv`. These three files are the export. The database is only a copy that is built from them.

## 3. Step 2: check each file with a checksum

A **checksum** is a long code calculated from the bytes of a file. `get_data.py` uses SHA-256. It knows the correct code of each file, and it compares. Run the cell.

In [ ]:
data = Path("data/raw/tickets.csv").read_bytes()
print("expected:", FILES["tickets.csv"])
print("found:   ", hashlib.sha256(data).hexdigest())

> **Check.** You should see the same code twice. The file is correct.

> **Predict.** Change only the first byte of the file's content. How much of the code changes? Run the cell: it changes the bytes in memory only, not the file.

In [ ]:
damaged = b"X" + data[1:]
print("found:   ", hashlib.sha256(damaged).hexdigest())

> **Check.** You should see a completely different code. If one byte is different, the code is different. So `get_data.py` stops with `... has the wrong checksum` if a download is damaged.

## 4. Step 3: load the CSV files into tables

`get_data.py` creates the four tables (`customers`, `tickets`, `outcomes` and `snapshot`) and inserts every line of each CSV file as a row. **An empty field becomes NULL.** Run the cell to see the CSV header and the line of ticket T-10064.

In [ ]:
lines = Path("data/raw/tickets.csv").read_text(encoding="utf-8").splitlines()
print(lines[0])
print(next(line for line in lines if line.startswith("T-10064")))

> **Check.** The line of T-10064 has two commas side by side after the ID (`T-10064,,2026-…`): its `customer_id` is empty. Its `order_value` is empty too.

> **Predict.** What does the database have for these two values? Run the cell.

In [ ]:
sql("""
SELECT ticket_id, customer_id, order_value
FROM tickets
WHERE ticket_id = 'T-10064';
""")

> **Check.** You should see `NULL` for `customer_id` and for `order_value`.

## 5. Step 4: open the database read-only

The setup cell opened the database as `con` in **read-only** mode, as `q.py` does. A query can read, but it cannot change anything.

> **Predict.** What happens if you try to delete all tickets? Run the cell.

In [ ]:
try:
    con.execute("DELETE FROM tickets")
except sqlite3.Error as error:
    print("SQL error:", error)
sql("SELECT COUNT(*) AS ticket_rows FROM tickets;")

> **Check.** You should see `SQL error: attempt to write a readonly database`, and still `1417` ticket rows. Nothing changed.

## 6. Run the real scripts

Now download the two scripts that you use on your computer, into `ticket-data`. The next cell also defines `terminal()`, which runs a Python script as a terminal does, and prints its output. Here, `terminal("q.py", "...")` is the same as `python3 q.py "..."` in your terminal. Run the cell.

In [ ]:
for script in ["get_data.py", "q.py"]:
    if not Path(script).exists():
        download(LABS + script, Path(script))
    print("downloaded", script)

def terminal(*args, cwd="."):
    """Run a Python script with these arguments, as a terminal does, and print its output."""
    result = subprocess.run([sys.executable, *args], capture_output=True, text=True, cwd=cwd)
    print(result.stdout + result.stderr, end="")

> **Check.** You should see `downloaded get_data.py` and `downloaded q.py`.

`larkfield.db` already exists here, because the setup cell built it.

> **Predict.** Does `get_data.py` replace it? Run the cell.

In [ ]:
terminal("get_data.py")

> **Check.** You should see `larkfield.db already exists. To build it again from the CSV files, run: python get_data.py --reset`. The script never replaces the database by accident.

The **reset** deletes `larkfield.db` and builds it again from the CSV files. Use it when you are not sure that the database is still the original. Run the cell.

In [ ]:
con.close()  # close the notebook's connection first, so the script can delete the file
terminal("get_data.py", "--reset")
con = sqlite3.connect("file:larkfield.db?mode=ro", uri=True)  # open the new database, read-only

> **Check.** You should see `Deleted larkfield.db.`, one line per table, and `Built larkfield.db. Check: customers 240, tickets 1417, outcomes 1874.` There is no `Downloading` line: the CSV files are already in `data/raw/`, and their checksums are correct.

Now ask the database a first question with `q.py`. Run the cell.

In [ ]:
terminal("q.py", "SELECT COUNT(*) FROM tickets")

> **Check.** You should see `1417` and `1 row`. This is the same number as in the browser boxes on the lesson page.

`q.py` opens the database read-only too. Run the cell.

In [ ]:
terminal("q.py", "DELETE FROM tickets")

> **Check.** You should see `SQL error: attempt to write a readonly database`.

> **On your computer.** The same commands are in [Build the database](https://learning.nextia-ai.com/courses/sql/m01/set-up-a-practice-database/#build-the-database) and in [The database is protected, and you can reset it](https://learning.nextia-ai.com/courses/sql/m01/set-up-a-practice-database/#the-database-is-protected-and-you-can-reset-it).

## 7. Run a saved query

A query saved in a file can be run again, compared and reviewed. The course numbers the files in `queries/`. The cell starts with `%%writefile`: it saves the rest of the cell as a file, as your code editor does. Run the cell.

In [ ]:
Path("queries").mkdir(exist_ok=True)

In [ ]:
%%writefile queries/01-first-look.sql
-- The first five tickets in the export
SELECT ticket_id, customer_id, created_at, team
FROM tickets
LIMIT 5;

> **Check.** You should see `Writing queries/01-first-look.sql` (or `Overwriting …` the second time).

A line that starts with `--` is a **comment**. SQL does not run it. The query names its columns instead of `SELECT *`, so it gives the same columns next month, even if the export adds a column.

> **Predict.** How many rows does `q.py` print? Run the cell.

In [ ]:
terminal("q.py", "queries/01-first-look.sql")

> **Check.** You should see 5 rows, from T-10001 to T-10005, and `5 rows`.

`q.py` prints at most 20 rows, then the total. Add `--all` to print every row. Run the cell and look at the last line.

In [ ]:
terminal("q.py", "SELECT ticket_id FROM tickets")

> **Check.** The last line is `1417 rows (showing 20; add --all to see every row)`.

> **On your computer.** Make the file in VS Code and run it in the terminal, as in [Run a saved query](https://learning.nextia-ai.com/courses/sql/m01/set-up-a-practice-database/#run-a-saved-query).

## 8. Failure case: `larkfield.db is not in this folder`

`q.py` looks for `larkfield.db` in the **working directory**: the folder that the terminal is in. The next cell runs `q.py` from the folder above `ticket-data`, as if you typed `python3 ticket-data/q.py …` in the `projects` folder.

> **Predict.** Does the query run? Run the cell.

In [ ]:
terminal("ticket-data/q.py", "SELECT COUNT(*) FROM tickets", cwd="..")

> **Check.** You should see `larkfield.db is not in this folder. Run the script from your project folder, or build the database first: python get_data.py`.

**Fix:** on your computer, run `pwd` to see the working directory, go to the project folder with `cd ~/projects/ticket-data`, and run the query again.

**In AI work:** scripts that use relative paths depend on the working directory. A script that finds no file and stops is good. A script that quietly makes a new, empty database is much worse.

## 9. Module practice: read the schema in SQL

SQLite describes each table with `pragma_table_info`. You use it like a table: one row for each column. Run the cell.

In [ ]:
sql("""
SELECT cid, name, type, pk
FROM pragma_table_info('tickets');
""")

> **Check.** You should see 11 rows, from `ticket_id` to `closed_at`. `cid` is the position of the column, from 0. `pk` is `1` for a column in a `PRIMARY KEY` constraint, and `0` otherwise. Every `pk` is `0`: the table has no primary key constraint, so you check the keys with queries.

Then count the rows of all three tables in one query. `UNION ALL` puts the results of several queries under each other. Save it as `queries/03-counts.sql`, and run it with `q.py`.

In [ ]:
%%writefile queries/03-counts.sql
-- Rows in each table of the export
SELECT 'customers' AS table_name, COUNT(*) AS row_count FROM customers
UNION ALL
SELECT 'tickets', COUNT(*) FROM tickets
UNION ALL
SELECT 'outcomes', COUNT(*) FROM outcomes;

In [ ]:
terminal("q.py", "queries/03-counts.sql")

> **Check.** You should see `customers 240`, `tickets 1417` and `outcomes 1874`: the same as the `Check:` line of `get_data.py`. So the load is complete.

### Your turn: the schema of `outcomes` and `customers`

> **Your turn.** Write a query that shows the `name` and the `type` of each column of `outcomes`. Then run the check cell. On your computer, save it as `queries/04-schema-outcomes.sql`.

In [ ]:
schema_outcomes = """
-- Show the name and the type of each column of outcomes
"""
sql(schema_outcomes)

In [ ]:
check(schema_outcomes, rows=6, columns=2, expected="4502df3cd333cb02")

> **Your turn.** Do the same for `customers`. On your computer, save it as `queries/05-schema-customers.sql`.

In [ ]:
schema_customers = """
-- Show the name and the type of each column of customers
"""
sql(schema_customers)

In [ ]:
check(schema_customers, rows=4, columns=2, expected="90d26234943346e8")

### Your turn: write your notes

> **Your turn.** Complete `notes.md`: one line for each table, with its row count, its grain (what one row is) and its primary key. For `tickets`, also give the number of different tickets. Replace each `...`, run the cell, then run the check cell. The check looks at the numbers and the key names; compare your words for the grain with the lesson.

In [ ]:
%%writefile notes.md
# Larkfield export of 2026-07-01 06:00

- customers: ... rows. One row = ... Primary key: ...
- tickets: ... rows, ... tickets. One row = ... Primary key: ...
- outcomes: ... rows. One row = ... Primary key: ...

In [ ]:
# Check notes.md: one line per table, with its row count and its primary key.
# The check does not judge your words for the grain: compare them with the lesson.
needed = {"customers": ["240", "customer_id"],
          "tickets": ["1417", "1403", "ticket_id"],
          "outcomes": ["1874", "outcome_id"]}
lines = Path("notes.md").read_text(encoding="utf-8").splitlines()
problems = []
for table, words in needed.items():
    line = next((l for l in lines if l.lstrip("- ").startswith(table)), "")
    if not line:
        problems.append(f"there is no line that starts with {table}")
    elif "..." in line or not all(w in line.replace(",", "") for w in words):
        problems.append(f"the line for {table} needs its numbers and its primary key")
print("Check passed." if not problems else "Not yet: " + "; ".join(problems) + ".")

## 10. What you do on your computer

The notebook did each step here. To make the project on your own computer, follow the lesson page in a terminal:

1. Open a terminal. If you need help, see [Get a terminal](https://learning.nextia-ai.com/courses/python/m01/files-and-terminal/#get-a-terminal) in the Python course.
2. Make `~/projects/ticket-data`, download `get_data.py` and `q.py`, and build the database: [Build the database](https://learning.nextia-ai.com/courses/sql/m01/set-up-a-practice-database/#build-the-database).
3. Save and run `queries/01-first-look.sql`: [Run a saved query](https://learning.nextia-ai.com/courses/sql/m01/set-up-a-practice-database/#run-a-saved-query).
4. Do the module practice again there, with `q.py`: [Module practice](https://learning.nextia-ai.com/courses/sql/m01/set-up-a-practice-database/#module-practice).
5. If the database is changed or broken, reset it: [The database is protected, and you can reset it](https://learning.nextia-ai.com/courses/sql/m01/set-up-a-practice-database/#the-database-is-protected-and-you-can-reset-it).

The lesson page also has [SQLite and PostgreSQL](https://learning.nextia-ai.com/courses/sql/m01/set-up-a-practice-database/#sqlite-and-postgresql), a table of the differences that you meet later.

## Check your understanding and recap

Answer the **knowledge checks** and the **module check** on the [lesson page](https://learning.nextia-ai.com/courses/sql/m01/set-up-a-practice-database/#module-check). They count toward your certificate when you are signed in and enrolled.

In this lesson you learned what `get_data.py` does: it downloads the export, checks each file with a checksum, and loads it into `larkfield.db`. `q.py` runs a query from a file and opens the database read-only. `get_data.py --reset` builds the database again from the CSV files. `pragma_table_info` shows the schema of a table, and the export has no key constraints.

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Select and filter](https://learning.nextia-ai.com/courses/sql/m02/select-and-filter/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/sql/m01/set-up-a-practice-database/)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.